# Q-learning et SARSA sur CartPole : on-policy vs off-policy

| Propriété | Valeur |
|---|---|
| Type | Model-free |
| Approche | Value-based |
| Représentation | Tabulaire (Q-table) |

On présente ici deux algorithmes jumeaux de **TD control** : Q-learning et SARSA. Ils partagent la même représentation (une table de Q-valeurs), la même politique d'exploration ($\varepsilon$-greedy), et le même environnement (CartPole-v1). Une seule ligne diffère dans la mise à jour : le choix du bootstrap. Cette différence, minime en apparence, traduit une distinction fondamentale entre apprentissage **off-policy** et **on-policy**.

On implémente les deux algorithmes, on les entraîne côte à côte avec les mêmes hyperparamètres, et on compare leurs courbes d'apprentissage et leurs politiques finales.

**Références** : [Watkins, 1989](https://link.springer.com/article/10.1007/BF00992698) (Q-learning) ; [Rummery & Niranjan, 1994](https://mi.eng.cam.ac.uk/reports/svr-ftp/auto-pdf/rummery_tr166.pdf) (SARSA) ; [Sutton & Barto, *Reinforcement Learning: An Introduction*](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf) , chapitres 6.4–6.5.

## On-policy vs off-policy

La distinction centrale entre Q-learning et SARSA tient à la relation entre la politique suivie pendant l'entraînement et la politique évaluée par les Q-valeurs.

**Off-policy (Q-learning)** : la cible de mise à jour utilise $\max_{a'} Q(s', a')$, c'est-à-dire la Q-valeur de l'action optimale dans l'état suivant, indépendamment de l'action que l'agent a réellement choisie. On peut donc apprendre la politique optimale tout en explorant de façon aléatoire.

**On-policy (SARSA)** : la cible utilise $Q(s', a')$ où $a'$ est l'action effectivement choisie par la politique $\varepsilon$-greedy. On apprend la valeur de ce que l'on fait réellement, exploration comprise.

Une analogie : Q-learning apprend le chemin optimal en regardant une carte. SARSA apprend de sa propre expérience de marche, détours compris.

La conséquence directe : SARSA est plus conservateur. Ses Q-valeurs intègrent le risque lié à l'exploration, ce qui le pousse vers des trajectoires plus sûres lorsque l'environnement contient des zones dangereuses.

In [ ]:
import random
from dataclasses import dataclass

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path
from IPython.display import Video, display


plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100


## CartPole-v1

L'environnement CartPole-v1 simule un pendule inversé sur un chariot mobile. L'observation est un vecteur continu de dimension 4 :

- Position du chariot
- Vitesse du chariot
- Angle du pendule par rapport à la verticale
- Vitesse angulaire du pendule

L'agent dispose de 2 actions discrètes : pousser le chariot vers la gauche ou vers la droite. La récompense est de $+1$ par step tant que le pendule reste droit et que le chariot ne sort pas du cadre. L'épisode se termine lorsque l'angle dépasse $\pm 12° (\pm 0.2095^{rad})$ ou que la position dépasse $\pm 2.4$.

Le seuil de résolution est de **500** sur 500 steps.

In [ ]:
env = gym.make("CartPole-v1")
print(f"Espace d'observation : {env.observation_space}")
print(f"Espace d'actions     : {env.action_space}")
obs, _ = env.reset(seed=42)
print(f"\nObservation initiale : {obs}")
print(f"  Position du chariot : {obs[0]:.4f}")
print(f"  Vitesse du chariot  : {obs[1]:.4f}")
print(f"  Angle du pendule    : {obs[2]:.4f} rad ({np.degrees(obs[2]):.2f}\u00b0)")
print(f"  Vitesse angulaire   : {obs[3]:.4f}")

# Episode aléatoire
total = 0
obs, _ = env.reset()
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    total += reward
    done = terminated or truncated
print(f"\nReward épisode aléatoire : {total}")
env.close()

## Discrétisation de l'espace d'observation: Q-table

Une Q-table est indexée par des entiers. Pour utiliser une Q-table avec CartPole, on doit convertir les observations continues en indices discrets.

On découpe chaque dimension en $n$ bins à l'aide de `torch.linspace` pour les bornes, puis on utilise `torch.bucketize` pour associer chaque valeur continue à son bin.

Ce découpage introduit une perte d'information : deux observations proches mais dans des bins différents sont traitées comme distinctes, et deux observations éloignées dans le même bin sont confondues. On revient sur cette limite (la malédiction de la dimensionnalité) en conclusion.

La Q-table a pour shape `(*bins, n_actions)`. Chaque entrée $Q(s, a)$ stocke l'estimation de la valeur de prendre l'action $a$ dans l'état discret $s$.

### Pourquoi utiliser une Q-table ?

Intuition : la Q-table est une **mémoire d'expérience**.

- Une case = "si je suis dans cet état et que je fais cette action, qu'est-ce que ça me rapporte à long terme ?"
- À force d'essais, les bonnes décisions montent en valeur, les mauvaises baissent.
- La politique finale devient simplement : prendre l'action avec la plus grande valeur dans la ligne de l'état courant.

Intérêt : pour un problème discret de taille modérée, c'est :

- Très simple à implémenter et à debugger
- Stable (pas d'approximation par réseau de neurones)
- Interprétable (on peut inspecter directement les valeurs)

On initialise souvent les valeurs avec de petits nombres aléatoires (plutôt que des zéros) pour briser la symétrie initiale entre les actions.

### Malédiction de la dimensionnalité (avec chiffres)

Nombre de valeurs à stocker :

$$
\#Q = \left(\prod_{i=1}^{d} n_i\right) \times |\mathcal{A}|
$$

où $d$ est le nombre de dimensions d'état, $n_i$ le nombre de bins de la dimension $i$, et $|\mathcal{A}|$ le nombre d'actions.

Exemples :

- Cas actuel (8 bins sur 4 dimensions, 2 actions) :
  $$8^4 \times 2 = 8192$$
- Si on passe à 30 bins sur 4 dimensions, 2 actions :
  $$30^4 \times 2 = 1\,620\,000$$
- Si l'état avait 6 dimensions à 30 bins (2 actions) :
  $$30^6 \times 2 = 1\,458\,000\,000$$

Le vrai coût n'est pas seulement mémoire, c'est surtout **l'exploration** :

- Plus il y a de couples $(s,a)$, plus il faut de transitions pour estimer correctement chaque valeur.
- Exemple : 1000 épisodes de 200 steps donnent $200\,000$ transitions.
- Réparties sur $1\,620\,000$ couples $(s,a)$, cela fait en moyenne environ $0.12$ visite par case (distribution uniforme idéale, donc très optimiste).

Conclusion : la Q-table est parfaite pour comprendre et prototyper, mais elle passe mal à l'échelle quand la granularité ou la dimension augmente. C'est précisément ce qui motive les méthodes avec approximation de fonction (DQN, actor-critic, etc.).

In [ ]:
# Configuration locale de cette illustration, indépendante du futur entraînement.
DEMO_BINS = (8, 8, 8, 8)
DEMO_BOUNDS = [
    (-4.8, 4.8),
    (-3.0, 3.0),
    (-0.418, 0.418),
    (-3.0, 3.0),
]

bin_edges = [
    torch.linspace(low, high, n + 1)[1:-1]
    for (low, high), n in zip(DEMO_BOUNDS, DEMO_BINS)
]

def discretize(obs):
    """Convertir une observation continue en tuple d'indices discrets."""
    return tuple(
        int(torch.bucketize(torch.tensor(obs[i]), bin_edges[i]).item())
        for i in range(4)
    )

# Test
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=42)
state = discretize(obs)
print(f"Observation : {obs}")
print(f"État discret : {state}")
print(f"Nombre total d'états : {np.prod(DEMO_BINS)} ({' × '.join(map(str, DEMO_BINS))})")
print(f"Q-valeurs totales    : {np.prod(DEMO_BINS) * 2}")
env.close()

In [ ]:
env = gym.make("CartPole-v1")
visited = set()
obs_samples = []
state_samples = []
for episode in range(20):
    obs, _ = env.reset(seed=42 + episode)
    done = False
    while not done:
        state = discretize(obs)
        visited.add(state)
        obs_samples.append(obs.copy())
        state_samples.append(state)
        action = env.action_space.sample()
        obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
env.close()

obs_array = np.asarray(obs_samples)
state_array = np.asarray(state_samples)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(obs_array[:, 0], obs_array[:, 2], s=10, alpha=0.25, color="tab:blue")
for edge in bin_edges[0]:
    axes[0].axvline(edge.item(), color="tab:gray", linestyle=":", linewidth=0.8)
for edge in bin_edges[2]:
    axes[0].axhline(edge.item(), color="tab:gray", linestyle=":", linewidth=0.8)
axes[0].set_xlabel("Position du chariot")
axes[0].set_ylabel("Angle du pendule")
axes[0].set_title("Observations continues et frontières de bins")
axes[0].grid(True, alpha=0.2)

heatmap = np.zeros((DEMO_BINS[2], DEMO_BINS[0]), dtype=np.int32)
for cart_bin, _, angle_bin, _ in state_array:
    heatmap[angle_bin, cart_bin] += 1
im = axes[1].imshow(heatmap, origin="lower", aspect="auto", cmap="Blues")
axes[1].set_xlabel("Bin position")
axes[1].set_ylabel("Bin angle")
axes[1].set_title("États discrétisés réellement visités")
plt.colorbar(im, ax=axes[1], shrink=0.85, label="Visites")

plt.tight_layout()
plt.show()
print(f"États uniques visités en 20 épisodes : {len(visited)} / {np.prod(DEMO_BINS)}")

**Lecture.** Les observations aléatoires n'occupent qu'une petite fraction de la grille : plusieurs bins restent vides, tandis que la zone proche de l'équilibre concentre les visites. Une discrétisation uniforme gaspille donc une partie de sa capacité loin des états courants et fusionne encore des observations différentes dans un même bin. C'est le compromis tabulaire central : davantage de bins réduit cette **aliasing**, mais agrandit la table et demande davantage d'expérience pour la couvrir.

In [ ]:
q_table = torch.rand((*DEMO_BINS, 2)) * 0.01
print(f"Shape Q-table : {q_table.shape}")
print(f"Min : {q_table.min():.6f}, Max : {q_table.max():.6f}, Moyenne : {q_table.mean():.6f}")

center = tuple(n // 2 for n in DEMO_BINS)
print(f"\nQ-valeurs pour l'état central {center} :")
print(f"  Action 0 (gauche) : {q_table[center + (0,)].item():.6f}")
print(f"  Action 1 (droite) : {q_table[center + (1,)].item():.6f}")

## Exploration vs Exploitation

Au début, on pourrait être tenté de choisir simplement l'action avec la plus grande Q-valeur, donc une politique **greedy** :

$$a = \arg\max_{a'} Q(s, a')$$

L'intuition est simple : si une action semble meilleure, pourquoi ne pas la prendre tout le temps ? Le problème est que cette stratégie ne laisse presque aucune place à l'apprentissage. Si les Q-valeurs sont encore mal estimées au début, l'agent peut rester bloqué sur une action qui paraît bonne seulement parce qu'il n'a pas assez essayé les autres.

C'est exactement le dilemme **exploration vs exploitation** :

- **Exploitation** : utiliser ce que l'on croit déjà savoir pour maximiser le gain immédiat.
- **Exploration** : tester d'autres actions pour recueillir de l'information et corriger ses estimations.

On introduit donc une politique $\varepsilon$-greedy, qui mélange les deux :

$$a = \begin{cases} \text{action aléatoire} & \text{avec probabilité } \varepsilon \\ \arg\max_{a'} Q(s, a') & \text{avec probabilité } 1 - \varepsilon \end{cases}$$

Avec cette règle, l'agent exploite la meilleure action connue la plupart du temps, mais garde une porte ouverte pour découvrir une action encore meilleure. C'est un peu comme suivre une recette connue tout en acceptant de goûter parfois un ingrédient nouveau pour voir s'il améliore le plat.

On fait décroître $\varepsilon$ au fil des épisodes avec un decay exponentiel :

$$\varepsilon_{t+1} = \max(\varepsilon_{\min},\; \varepsilon_t \cdot d)$$

L'idée est logique : au début, on explore beaucoup parce qu'on connaît mal l'environnement. Ensuite, quand les estimations deviennent plus fiables, on réduit l'exploration pour exploiter davantage ce qu'on a appris.

Cette politique est partagée par Q-learning et SARSA. Seule la règle de mise à jour des Q-valeurs diffère entre les deux algorithmes.

In [ ]:
def select_action_greedy(q_table, state):
    """Choisir l'action avec la Q-valeur maximale (ties broken randomly)."""
    values = q_table[state]
    max_val = torch.max(values)
    best = torch.nonzero(values == max_val, as_tuple=False).flatten()
    return int(best[np.random.randint(len(best))].item())

def select_action_epsilon_greedy(q_table, state, epsilon, rng):
    """epsilon-greedy : exploration avec probabilité epsilon, exploitation sinon."""
    if rng.random() < epsilon:
        return int(rng.integers(q_table.shape[-1]))
    return select_action_greedy(q_table, state)

# Test : distribution sur 10 000 tirages à epsilon = 0.2
rng = np.random.default_rng(42)
q_table_test = torch.zeros((*DEMO_BINS, 2))
test_state = (4, 4, 4, 4)
q_table_test[test_state + (1,)] = 1.0  # action 1 = meilleure

counts = {0: 0, 1: 0}
for _ in range(10_000):
    a = select_action_epsilon_greedy(q_table_test, test_state, 0.2, rng)
    counts[a] += 1
print(f"\u03b5 = 0.2, Q[s,0] = 0, Q[s,1] = 1")
print(f"Action 0 : {counts[0]/100:.1f}% (attendu ~10%)")
print(f"Action 1 : {counts[1]/100:.1f}% (attendu ~90%)")

# Decay
epsilons = []
eps = 1.0
for _ in range(500):
    epsilons.append(eps)
    eps = max(0.01, eps * 0.995)

plt.figure(figsize=(8, 3))
plt.plot(epsilons)
plt.xlabel("\u00c9pisode")
plt.ylabel("\u03b5")
plt.title("D\u00e9croissance exponentielle de \u03b5")
plt.tight_layout()
plt.show()

**Lecture.** Avec $\varepsilon=0.2$, la branche aléatoire choisit elle-même entre deux actions : l'action non greedy apparaît donc environ $0.2/2=10\%$ du temps, et non $20\%$. La courbe montre ensuite que l'exploration décroît progressivement jusqu'à un plancher ; l'agent exploite de plus en plus sa table sans devenir totalement incapable de visiter une alternative.

## Équation de Bellman et apprentissage TD

L'équation de Bellman d'optimalité relie la Q-valeur optimale d'un couple $(s, a)$ à celles de l'état suivant :

$$Q^*(s,a) = \mathbb{E}\left[r + \gamma \max_{a'} Q^*(s', a')\right]$$

On ne connaît pas $Q^*$ directement, mais on peut s'en approcher itérativement. À chaque transition, on calcule une erreur temporelle (TD error) et on ajuste la Q-valeur dans cette direction :

$$\delta = \text{target} - Q(s, a) \qquad Q(s,a) \leftarrow Q(s,a) + \alpha \, \delta$$

La question est : quel target utiliser ? C'est là que Q-learning et SARSA divergent.

## Les deux règles de mise à jour

**Q-learning** (off-policy) :

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ \underbrace{\underbrace{r}_{\text{reward}} + \underbrace{\gamma}_{\text{discount factor}} \underbrace{\max_{a'} Q(s', a')}_{\text{off-policy}} - Q(s, a)}_{\delta} \right]$$

**SARSA** (on-policy) :

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \, \underset{\text{on-policy}}{\underbrace{Q(s', a')}} - Q(s, a) \right]$$

Le nom SARSA vient du quintuplet $(s, a, r, s', a')$ : **S**tate, **A**ction, **R**eward, **S**tate, **A**ction. Ce sont les cinq éléments nécessaires pour effectuer une mise à jour. Q-learning, en revanche, n'a besoin que de $(s, a, r, s')$ : il calcule le max lui-même.

Quand $a' = \arg\max_{a'} Q(s', a')$ (l'agent exploite), les deux mises à jour sont identiques. La différence n'apparaît que lorsque $a'$ est une action d'exploration.

In [ ]:
def q_learning_update(q_table, state, action, reward, next_state, done, alpha, gamma):
    """Q-learning : utilise max Q(s', a') comme bootstrap."""
    current = q_table[state + (action,)].item()
    bootstrap = 0.0 if done else torch.max(q_table[next_state]).item()
    target = reward + gamma * bootstrap
    td_error = target - current
    q_table[state + (action,)] += alpha * td_error
    return td_error

def sarsa_update(q_table, state, action, reward, next_state, next_action, done, alpha, gamma):
    """SARSA : utilise Q(s', a') comme bootstrap (a' = action réellement choisie)."""
    current = q_table[state + (action,)].item()
    bootstrap = 0.0 if done else q_table[next_state + (next_action,)].item()
    target = reward + gamma * bootstrap
    td_error = target - current
    q_table[state + (action,)] += alpha * td_error
    return td_error

In [ ]:
# Même Q-table pour les deux
q_ql = torch.zeros((3, 3, 3, 3, 2))
q_sa = q_ql.clone()

state = (1, 1, 1, 1)
next_state = (2, 2, 2, 2)
action = 0
reward = 1.0
alpha, gamma = 0.5, 0.9

# Q-valeurs du next_state : action 0 = 2.0, action 1 = 5.0
q_ql[next_state + (0,)] = 2.0
q_ql[next_state + (1,)] = 5.0
q_sa[next_state + (0,)] = 2.0
q_sa[next_state + (1,)] = 5.0

# next_action = 0 (pas le argmax !)
next_action = 0

td_ql = q_learning_update(q_ql, state, action, reward, next_state, False, alpha, gamma)
td_sa = sarsa_update(q_sa, state, action, reward, next_state, next_action, False, alpha, gamma)

print(f"Q[next_state, 0] = 2.0,  Q[next_state, 1] = 5.0")
print(f"next_action = {next_action} (pas le argmax)")
print()
print(f"Q-learning : target = {reward} + {gamma} \u00d7 max(2.0, 5.0) = {reward + gamma * 5.0}")
print(f"  Q[s,a] apr\u00e8s update = {q_ql[state + (action,)].item():.4f}")
print()
print(f"SARSA      : target = {reward} + {gamma} \u00d7 Q[s', a'=0] = {reward + gamma * 2.0}")
print(f"  Q[s,a] apr\u00e8s update = {q_sa[state + (action,)].item():.4f}")
print()
print(f"Diff\u00e9rence : {abs(q_ql[state + (action,)].item() - q_sa[state + (action,)].item()):.4f}")

# Cas next_action == argmax : les deux sont identiques
q_ql2 = torch.zeros((3, 3, 3, 3, 2))
q_sa2 = q_ql2.clone()
q_ql2[next_state + (0,)] = 2.0
q_ql2[next_state + (1,)] = 5.0
q_sa2[next_state + (0,)] = 2.0
q_sa2[next_state + (1,)] = 5.0

q_learning_update(q_ql2, state, action, reward, next_state, False, alpha, gamma)
sarsa_update(q_sa2, state, action, reward, next_state, 1, False, alpha, gamma)  # next_action = argmax = 1
print(f"\nCas next_action = argmax = 1 :")
print(f"  Q-learning = {q_ql2[state + (action,)].item():.4f}")
print(f"  SARSA      = {q_sa2[state + (action,)].item():.4f}")
print(f"  Identiques : {q_ql2[state + (action,)].item() == q_sa2[state + (action,)].item()}")

### Pourquoi les Q-valeurs de Q-learning montent souvent plus haut que celles de SARSA ?

Deux mécanismes se combinent :

1. **Q-learning vise la meilleure action imaginable au prochain état** via `max_a Q(s', a)` ; si une seule action paraît très bonne, toute la cible grimpe.
2. **SARSA vise l'action effectivement suivie par la politique ε-greedy** ; tant que l'exploration subsiste, la cible reste plus prudente.

Dans CartPole, cela ne produit pas forcément des politiques finales très différentes, parce que l'environnement récompense simplement la survie. En revanche, les **estimations intermédiaires** diffèrent déjà sur une transition jouet, puis sur une série de mises à jour répétées.

In [ ]:
n_updates = 20
alpha, gamma = 0.5, 0.9
state = (1, 1, 1, 1)
next_state = (2, 2, 2, 2)
action = 0
next_action = 0
reward = 1.0

q_ql = torch.zeros((3, 3, 3, 3, 2))
q_sa = q_ql.clone()
q_ql[next_state + (0,)] = 2.0
q_ql[next_state + (1,)] = 5.0
q_sa[next_state + (0,)] = 2.0
q_sa[next_state + (1,)] = 5.0

ql_values, sa_values = [], []
for _ in range(n_updates):
    ql_values.append(q_ql[state + (action,)].item())
    sa_values.append(q_sa[state + (action,)].item())
    q_learning_update(q_ql, state, action, reward, next_state, False, alpha, gamma)
    sarsa_update(q_sa, state, action, reward, next_state, next_action, False, alpha, gamma)

plt.figure(figsize=(8, 4))
plt.plot(ql_values, "o-", label=f"Q-learning (target = r + \u03b3\u00b7max = {reward + gamma * 5.0})")
plt.plot(sa_values, "s-", label=f"SARSA (target = r + \u03b3\u00b7Q[s',a'=0] = {reward + gamma * 2.0})")
plt.axhline(y=reward + gamma * 5.0, color="tab:blue", linestyle="--", alpha=0.3)
plt.axhline(y=reward + gamma * 2.0, color="tab:orange", linestyle="--", alpha=0.3)
plt.xlabel("Update")
plt.ylabel("Q(s, a)")
plt.title("Convergence : Q-learning vs SARSA (m\u00eame transition)")
plt.legend()
plt.tight_layout()
plt.show()

## Des briques visibles vers deux agents différents

Les composants restent identiques d'un algorithme à l'autre :

- une discrétisation `obs -> state` ;
- une politique `ε`-greedy pour choisir une action ;
- une règle TD qui transforme une transition en cible ;
- une boucle d'environnement qui collecte `state, action, reward, next_state`.

La seule bifurcation réelle tient au **bootstrap** utilisé pour estimer la suite :

```mermaid
flowchart LR
classDef default fill:none,stroke:#64748b,stroke-width:1.5px
classDef primary fill:none,stroke:#2563eb,stroke-width:2px
classDef secondary fill:none,stroke:#d97706,stroke-width:2px
classDef result fill:none,stroke:#16a34a,stroke-width:2px
classDef warning fill:none,stroke:#dc2626,stroke-width:2px
S["Transition observee"]:::primary --> QL["Q-learning prend le max au prochain etat"]:::primary
S --> SA["SARSA reutilise la prochaine action vraiment choisie"]:::secondary
QL --> T1["Cible off-policy"]:::result
SA --> T2["Cible on-policy"]:::result
```

On peut donc assembler deux agents qui partagent la même table et la même politique d'exploration, mais pas la même lecture du futur.

## Hyperparamètres d'entraînement

| Paramètre | Valeur |
|---|---|
| TRAIN_BINS | (12, 12, 12, 12) |
| $\alpha$ (learning rate) | 0.1 |
| $\gamma$ (discount) | 1.0 |
| $\varepsilon$ initial | 1.0 |
| $\varepsilon$ decay | 0.995 |
| $\varepsilon$ minimum | 0.01 |
| Épisodes | 500 |

On fixe tout sauf la règle d'update pour isoler l'effet on-policy vs off-policy.

On entraîne avec `rendu local` pour voir l'agent progresser.

In [ ]:
# Configuration de l'expérience : toutes les valeurs qui gouvernent le training sont ici.
SEED = 42
ENV_ID = "CartPole-v1"
RUN_Q_LEARNING = True
RUN_SARSA = True
EVAL_SEED = 0

TRAIN_BINS = (12, 12, 12, 12)
BOUNDS = [
    (-4.8, 4.8),
    (-3.0, 3.0),
    (-0.418, 0.418),
    (-3.0, 3.0),
]
ALPHA = 0.1
GAMMA = 1.0
EPSILON_START = 1.0
EPSILON_DECAY = 0.995
EPSILON_MIN = 0.01
N_EPISODES = 500
MAX_STEPS = 500

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

train_bin_edges = [
    torch.linspace(low, high, n + 1)[1:-1]
    for (low, high), n in zip(BOUNDS, TRAIN_BINS)
]

def discretize_train(obs):
    return tuple(
        int(torch.bucketize(torch.tensor(obs[i]), train_bin_edges[i]).item())
        for i in range(4)
    )

In [ ]:
@dataclass
class TabularControlConfig:
    alpha: float
    gamma: float

class QLearningAgent:
    def __init__(self, q_table, config: TabularControlConfig):
        self.q_table = q_table
        self.config = config

    def select_action(self, state, epsilon, rng):
        return select_action_epsilon_greedy(self.q_table, state, epsilon, rng)

    def select_greedy_action(self, state):
        return select_action_greedy(self.q_table, state)

    def learn_step(self, state, action, reward, next_state, done):
        return q_learning_update(
            self.q_table,
            state,
            action,
            reward,
            next_state,
            done,
            self.config.alpha,
            self.config.gamma,
        )

class SarsaAgent(QLearningAgent):
    def learn_step(self, state, action, reward, next_state, next_action, done):
        return sarsa_update(
            self.q_table,
            state,
            action,
            reward,
            next_state,
            next_action,
            done,
            self.config.alpha,
            self.config.gamma,
        )

def train_q_learning_episode(agent, env, epsilon, episode_seed, rng):
    obs, _ = env.reset(seed=episode_seed)
    state = discretize_train(obs)
    total_reward = 0.0

    for _ in range(MAX_STEPS):
        action = agent.select_action(state, epsilon, rng)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = discretize_train(obs)
        agent.learn_step(state, action, reward, next_state, done)
        total_reward += reward
        state = next_state
        if done:
            break

    return total_reward

def train_sarsa_episode(agent, env, epsilon, episode_seed, rng):
    obs, _ = env.reset(seed=episode_seed)
    state = discretize_train(obs)
    action = agent.select_action(state, epsilon, rng)
    total_reward = 0.0

    for _ in range(MAX_STEPS):
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = discretize_train(obs)
        next_action = agent.select_action(next_state, epsilon, rng)
        agent.learn_step(state, action, reward, next_state, next_action, done)
        total_reward += reward
        state = next_state
        action = next_action
        if done:
            break

    return total_reward

In [ ]:
config = TabularControlConfig(alpha=ALPHA, gamma=GAMMA)

ql_env = gym.make(ENV_ID, render_mode="none")
ql_rng = np.random.default_rng(SEED)
q_learning_agent = QLearningAgent(torch.rand((*TRAIN_BINS, 2)) * 0.01, config)

ql_rewards = []
ql_epsilons = []
epsilon = EPSILON_START

if RUN_Q_LEARNING:
    for episode in range(1, N_EPISODES + 1):
        reward = train_q_learning_episode(q_learning_agent, ql_env, epsilon, SEED + episode, ql_rng)
        ql_rewards.append(reward)
        ql_epsilons.append(epsilon)
        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

        if episode % 50 == 0:
            avg = np.mean(ql_rewards[-50:])
            print(f"[Q-learning] Épisode {episode:4d} | Reward {reward:6.0f} | Moy 50 : {avg:6.1f} | ε = {epsilon:.4f}")

ql_env.close()
ql_q_table = q_learning_agent.q_table

sarsa_env = gym.make(ENV_ID, render_mode="none")
sarsa_rng = np.random.default_rng(SEED)
sarsa_agent = SarsaAgent(torch.rand((*TRAIN_BINS, 2)) * 0.01, config)

sarsa_rewards = []
sarsa_epsilons = []
epsilon = EPSILON_START

if RUN_SARSA:
    for episode in range(1, N_EPISODES + 1):
        reward = train_sarsa_episode(sarsa_agent, sarsa_env, epsilon, SEED + episode, sarsa_rng)
        sarsa_rewards.append(reward)
        sarsa_epsilons.append(epsilon)
        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

        if episode % 50 == 0:
            avg = np.mean(sarsa_rewards[-50:])
            print(f"[SARSA]      Épisode {episode:4d} | Reward {reward:6.0f} | Moy 50 : {avg:6.1f} | ε = {epsilon:.4f}")

sarsa_env.close()
sarsa_q_table = sarsa_agent.q_table

In [ ]:
def moving_average(data, window=50):
    return [np.mean(data[max(0, i - window + 1):i + 1]) for i in range(len(data))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1 : rewards
axes[0].plot(ql_rewards, alpha=0.15, color="tab:blue")
axes[0].plot(sarsa_rewards, alpha=0.15, color="tab:orange")
axes[0].plot(moving_average(ql_rewards), label="Q-learning", color="tab:blue", linewidth=2)
axes[0].plot(moving_average(sarsa_rewards), label="SARSA", color="tab:orange", linewidth=2)
axes[0].set_xlabel("\u00c9pisode")
axes[0].set_ylabel("Reward")
axes[0].set_title("Reward par \u00e9pisode")
axes[0].legend()

# Panel 2 : epsilon
axes[1].plot(ql_epsilons, label="Q-learning", color="tab:blue")
axes[1].plot(sarsa_epsilons, label="SARSA", color="tab:orange", linestyle="--")
axes[1].set_xlabel("\u00c9pisode")
axes[1].set_ylabel("\u03b5")
axes[1].set_title("D\u00e9croissance de \u03b5")
axes[1].legend()

plt.tight_layout()
plt.show()

## Lecture approfondie des courbes

Les deux panneaux racontent deux histoires différentes.

1. **Reward par épisode.** La version brute est très bruitée, mais la moyenne mobile révèle que les deux agents trouvent rapidement une politique qui maintient le pendule en équilibre. Sur CartPole, la différence on-policy/off-policy est donc moins un débat de performance finale qu'un débat de **chemin d'apprentissage**.
2. **Décroissance de ε.** Comme le planning d'exploration est identique, les écarts observés dans la reward ne viennent pas d'un budget plus favorable pour l'un des deux agents. Ils viennent uniquement de la manière dont chaque update interprète l'état suivant.

En pratique, Q-learning profite souvent d'une cible un peu plus optimiste, alors que SARSA internalise immédiatement le coût de ses futures actions exploratoires. C'est précisément cette prudence qui devient cruciale dans les environnements où une seule mauvaise action coûte très cher.

## Cliff Walking : là où la différence compte

L'exemple classique de Sutton & Barto (section 6.5) illustre la différence entre Q-learning et SARSA sur un environnement avec pénalité asymétrique : le Cliff Walking. Un agent se déplace sur une grille avec une falaise le long du bord inférieur. Tomber de la falaise donne une pénalité de $-100$ et ramène au départ.

Q-learning apprend le chemin optimal qui longe la falaise (le plus court), mais l'agent tombe fréquemment pendant l'entraînement parce que l'exploration $\varepsilon$-greedy le pousse parfois dans le vide.

SARSA apprend un chemin plus long mais plus sûr, qui passe loin de la falaise. Ses Q-valeurs intègrent le risque de chute lié à l'exploration : les états proches du bord ont des valeurs basses parce que l'agent y a réellement chuté.

Sur CartPole, il n'y a pas d'équivalent de la falaise. L'épisode se termine quand le pendule tombe, mais la pénalité est la même partout (fin de la récompense). C'est pourquoi les deux algorithmes se comportent de façon similaire ici.

In [ ]:
cliff_grid = np.zeros((4, 12))
cliff_grid[3, 1:11] = -1
cliff_grid[3, 0] = 1
cliff_grid[3, 11] = 2

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

cmap = plt.cm.get_cmap("RdYlGn", 3)
axes[0].imshow(cliff_grid, cmap=cmap, vmin=-1, vmax=2)
for x in range(13):
    axes[0].axvline(x - 0.5, color="white", linewidth=1)
for y in range(5):
    axes[0].axhline(y - 0.5, color="white", linewidth=1)
axes[0].text(0, 3, "S", ha="center", va="center", color="black", fontsize=12, fontweight="bold")
axes[0].text(11, 3, "G", ha="center", va="center", color="black", fontsize=12, fontweight="bold")
axes[0].annotate("chemin prudent typique de SARSA", xy=(0.6, 2.2), xytext=(4.0, 0.6),
                 arrowprops=dict(arrowstyle="->", color="tab:orange", lw=2), color="tab:orange")
axes[0].annotate("raccourci risqué souvent valorisé par Q-learning", xy=(5.5, 3.0), xytext=(5.0, 1.2),
                 arrowprops=dict(arrowstyle="->", color="tab:blue", lw=2), color="tab:blue")
axes[0].set_title("Cliff Walking : pourquoi on-policy vs off-policy compte")
axes[0].set_xticks([])
axes[0].set_yticks([])

n_last = 50
ql_last = ql_rewards[-n_last:]
sa_last = sarsa_rewards[-n_last:]
metrics = [
    ("Moy. 50 derniers", np.mean(ql_last), np.mean(sa_last)),
    ("Meilleur épisode", max(ql_rewards), max(sarsa_rewards)),
    ("Dernier épisode", ql_rewards[-1], sarsa_rewards[-1]),
    ("Q-valeurs non nulles", (ql_q_table.abs() > 1e-6).sum().item(), (sarsa_q_table.abs() > 1e-6).sum().item()),
]

y = 0.95
axes[1].axis("off")
axes[1].text(0.0, y, "Comparatif final", fontsize=13, fontweight="bold", transform=axes[1].transAxes)
y -= 0.12
for label, q_val, s_val in metrics:
    axes[1].text(0.0, y, label, transform=axes[1].transAxes)
    axes[1].text(0.62, y, f"{q_val:>8.1f}" if isinstance(q_val, float) else f"{q_val:>8}", color="tab:blue", transform=axes[1].transAxes)
    axes[1].text(0.84, y, f"{s_val:>8.1f}" if isinstance(s_val, float) else f"{s_val:>8}", color="tab:orange", transform=axes[1].transAxes)
    y -= 0.1
axes[1].text(0.62, 0.08, "Q-learning", color="tab:blue", fontsize=11, transform=axes[1].transAxes)
axes[1].text(0.84, 0.08, "SARSA", color="tab:orange", fontsize=11, transform=axes[1].transAxes)
axes[1].text(0.0, -0.02, "CartPole ne punit pas fortement les prises de risque. Cliff Walking sert donc de loupe conceptuelle :\nla mise à jour off-policy cherche le meilleur futur possible, la mise à jour on-policy apprend le futur réellement vécu.", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

**Lecture.** CartPole récompense surtout la survie et distingue peu un chemin prudent d'un chemin risqué. Cliff Walking sert ici de loupe conceptuelle : Q-learning bootstrappe vers la meilleure action possible, alors que SARSA tient compte de l'action encore exploratoire qui sera réellement jouée. Cette illustration explique la différence de politique apprise ; elle ne prétend pas transformer les scores CartPole en preuve générale de supériorité.

## Récapitulatif

| | Q-learning | SARSA |
|---|---|---|
| Type de politique | Off-policy | On-policy |
| Bootstrap | $\max_{a'} Q(s', a')$ | $Q(s', a')$ |
| Apprend | La politique optimale | La politique actuelle ($\varepsilon$-greedy) |
| Convergence | Vers $Q^*$ (sous conditions) | Vers $Q^\pi$ (politique suivie) |
| Sensibilité à $\varepsilon$ | Moins sensible (prend le max) | Plus sensible (utilise $a'$ avec $\varepsilon$) |
| Stabilité | Peut surestimer (max bias) | Plus conservateur |
| Avec pénalités fortes | Chemin optimal mais risqué | Chemin sûr |
| Complexité par step | Identique | Identique |

## Évaluation greedy

L'évaluation finale fige maintenant $ε = 0$ pour répondre à une question simple : **que vaut la politique apprise une fois l'exploration retirée ?**

Ce test est utile, mais il faut le lire avec précaution. Pendant l'entraînement, surtout pour SARSA, l'agent a appris sous une politique encore partiellement exploratoire. Le mode greedy mesure donc la politique extraite de la table finale, pas exactement le comportement stochastique vécu pendant l'apprentissage.

In [ ]:
def evaluate_and_display_agent(
    agent,
    label,
    n_episodes=5,
    seed=EVAL_SEED,
    video_root="videos/01_tabular_cartpole",
):
    """Évalue un agent greedy et affiche le dernier replay vidéo enregistré."""
    rewards = []
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env = gym.make(ENV_ID, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    try:
        for i in range(n_episodes):
            obs, _ = env.reset(seed=seed + i)
            state = discretize_train(obs)
            total = 0.0
            done = False

            while not done:
                action = agent.select_greedy_action(state)
                obs, reward, terminated, truncated, _ = env.step(action)
                state = discretize_train(obs)
                total += reward
                done = terminated or truncated

            rewards.append(total)
            print(f"[{label}] Épisode {i + 1} : {total:.0f} steps")

    finally:
        env.close()

    print(f"[{label}] Moyenne : {np.mean(rewards):.1f}\n")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return rewards


print("--- Q-learning (greedy) ---")
ql_greedy = evaluate_and_display_agent(q_learning_agent, "Q-learning")

print("--- SARSA (greedy) ---")
sarsa_greedy = evaluate_and_display_agent(sarsa_agent, "SARSA")

## Conclusion et ponts vers la suite

Ce chapitre a construit la base de tout le RL value-based : apprendre une fonction de valeur par
**différence temporelle**, puis agir en choisissant l'action qui semble la meilleure tout en gardant
un peu d'exploration. Q-learning et SARSA partagent presque les mêmes ingrédients, mais leur différence
est fondamentale : Q-learning apprend une valeur **off-policy**, comme si l'agent suivait déjà la
politique gloutonne, tandis que SARSA apprend **on-policy**, avec les actions réellement prises par
sa politique exploratoire.

CartPole nous a aussi montré la limite du tabulaire. Pour utiliser une Q-table, il a fallu découper
un état continu en cases discrètes. Cela marche pour comprendre les équations, mais ce n'est pas une
solution scalable : plus l'état devient riche, plus la table explose, et plus la discrétisation perd
d'information.

La suite naturelle est donc **DQN**. On garde la même intuition TD et la même idée de politique
ε-greedy, mais on remplace la table par un réseau de neurones capable d'approximer $Q(s,a)$ directement
depuis un état continu. Ce changement paraît simple, pourtant il rend l'entraînement beaucoup moins
stable : le prochain notebook introduira donc les deux briques qui rendent DQN praticable, le
**replay buffer** et le **réseau cible**.